In [1]:
%run ../scripts/notebook_settings.py
import sgkit as sg
import xarray as xr
import glob

In [2]:
table_desc = "~/primatediversity/data/gVCFs_recalling_10_12_2024_metadata/plots/SupTable_Sample_Stats_wGT_QC.tsv"
metadata_path = "~/primatediversity/data/gVCFs_recalling_10_12_2024_metadata/"

metadata_table = pd.read_csv(table_desc, sep="\t")

metadata_20x_filt = metadata_table.loc[(metadata_table.finalQC != "fail")
                              & (metadata_table.cov_chrA >= 20)
                              & (metadata_table.remove_as_relative != True)
                              & (metadata_table.remove_manual != True)
                              & (~metadata_table.ID.str.startswith("SAMEA11633"))
                             ]

In [3]:
# Loading in the kuderna mutation data and new metadata

kuderna_data = pd.read_csv("../data/science.abn7829_data_s2.csv")
# I pick out the following columns
col_species_mut = ['SPECIES_BINOMIAL', 'GENUS', 'SPECIES', 'FAMILY', 'GENERATION_LENGTH', 'MU_PER_GENERATION',
       'MU_PER_YEAR', 'EFFECTIVE_POP_SIZE']
df_species_mut = kuderna_data[col_species_mut]

metadata_dir = "/home/eriks/primatediversity/data/gVCFs_recalling_10_12_2024_metadata/"
metadata_dirs = glob.glob(metadata_dir+"*_individuals.txt")

df_l = []
for d in metadata_dirs:
    # Identify IDs
    dir_metadata = pd.read_csv(d, sep="\t")
    df_l.append(dir_metadata)
all_inds = pd.concat(df_l)

In [4]:
def get_gamma_from_file(file):
    lines = !cat {file}
    return float([i for i in lines if 'gamma' in i][0].split(' ')[-1])
    
def get_LL_from_file(filename):
    with open(filename) as f:
        lines = f.readlines()
    return float([i for i in lines if 'likelihood' in i and 'final' in i][0].split(' ')[-1])

In [5]:
def find_best(file_list):
    best_param_file = None
    best_val = None
    # Go through the files and pick the 
    for p in file_list:
        LL = get_LL_from_file(p)
        if best_val == None:
            best_val = LL
            best_param_file = p
        if best_val < LL:
            best_val = LL
            best_param_file = p

    with open(best_param_file) as f:
        finallines = f.readlines()
    ztheta = float([i for i in finallines if 'theta' in i ][0].split(' ')[-1])
    zrho = float([i for i in finallines if 'rho' in i ][0].split(' ')[-1])
    zgamma = float([i for i in finallines if 'gamma' in i ][0].split(' ')[-1])
    file_name = best_param_file.split("/")[-1]
    zte = int(file_name.split("te")[1].split("_")[0])
    zts = int(file_name.split("ts")[1].split("_")[0])
    
    final_params = np.loadtxt(best_param_file)
    lambdaA_parameters = ",".join([str(x) for x in final_params[:,2]*ztheta/4])
    lambdaB_parameters = ",".join([str(x) for x in final_params[:,3]*ztheta/4])
    return zte, zts, ztheta, zrho, zgamma, lambdaA_parameters, lambdaB_parameters, best_param_file

In [20]:
ind_list = [x.split("/")[3] for x in glob.glob("../steps/cobraa/*/aut_PSMC_final_parameters.txt*")]
len(ind_list)

129

In [33]:
ind_list = [x.split("/")[3] for x in glob.glob("../steps/cobraa/*/aut_PSMC_D50_ts16_te28*")]
len(ind_list)

117

In [15]:
metadata_20x_filt.loc[metadata_20x_filt.ID.isin(ind_list)].species_genotyping.unique()

array(['Allenopithecus_nigroviridis_ssp', 'Alouatta_belzebul_ssp',
       'Alouatta_discolor_ssp', 'Alouatta_juara_ssp',
       'Alouatta_macconnelli_ssp', 'Alouatta_palliata_ssp',
       'Alouatta_seniculus_ssp', 'Ateles_belzebuth_ssp',
       'Ateles_chamek_ssp', 'Ateles_geoffroyi_ssp',
       'Ateles_marginatus_ssp', 'Brachyteles_hypoxanthus_ssp',
       'Callimico_goeldii_ssp', 'Callithrix_jacchus_ssp',
       'Cebuella_niveiventris_ssp', 'Mico_argentatus_ssp',
       'Mico_humeralifer_ssp', 'Cercopithecus_ascanius_ssp',
       'Cercopithecus_denti_ssp', 'Cercopithecus_hamlyni_ssp',
       'Cercopithecus_mitis_ssp', 'Cercopithecus_mona_ssp',
       'Cercopithecus_roloway_ssp', 'Cercopithecus_wolfi_ssp',
       'Cheirogaleus_major_ssp', 'Cheirogaleus_medius_ssp',
       'Cheirogaleus_sibreei_ssp', 'Colobus_angolensis_ssp',
       'Colobus_guereza_ssp', 'Daubentonia_madagascariensis_ssp',
       'Gorilla_beringei_ssp', 'Gorilla_gorilla_ssp',
       'Hylobates_agilis_ssp', 'Hylobates_

In [59]:
# Check of cobraa workflow to replicate individual choices.
s_l, i_l = [], []
for g in metadata_20x_filt.genus.unique()[:]:
    # Identify IDs
    species_metadata = metadata_20x_filt.loc[metadata_20x_filt.genus == g]
    #species_metadata = species_metadata.loc[species_metadata.cov_chrX >= 5]
    # I only want females as they have a diploid chrX.
    female_df = species_metadata.loc[(species_metadata.gSEX == "F")].sort_values(by="cov_chrA", ascending=False)
    # # Restricted analysis to females - below is the way to include males if you want to run non-X analysis.
    male_df = species_metadata.loc[~(species_metadata.ID.isin(female_df.ID))].sort_values(by="cov_chrA", ascending=False)
    sorted_df = pd.concat([female_df, male_df])
    if len(sorted_df) > 0:
        sorted_input = sorted_df
    else:
        print("No usable ind for", g)
        continue
    region_metadata = pd.read_csv(metadata_dir+g+"_regions_and_batches.txt",
                           sep="\t").sort_values(by="END")
    # Go through every unique genotype calling set.
    for gvcf_folder in sorted_input.species_genotyping.unique()[:]:
        # Pick the (currently one) best individuals in the metadata.
        species_input = sorted_input.loc[sorted_input.species_genotyping == gvcf_folder]
        if len(species_input) == 1 and species_input.gSEX.iloc[0] == "M":
            continue # Skip if theres only a single male
        picked_inds = sorted_input.loc[sorted_input.species_genotyping == gvcf_folder].ID.iloc[:1]
        i_l.append(picked_inds.iloc[0])

In [60]:
len(metadata_20x_filt.loc[metadata_20x_filt.ID.isin(i_l)].species_genotyping.unique())

198